#Install required library

In [2]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 6.5 MB/s eta 0:00:00


#Example, in here, we apply magic code in order to write it as file. If we run code on google colab kernel, it will produce error, as it will try to run Hadoop, we want to run on local machine :))

In [29]:
%%file test.py
from mrjob.job import MRJob
import math

class MRPrimeSum(MRJob):
    def mapper(self, _, line):
        # Parse input range (e.g., "1 10000000")
        start, end = map(int, line.strip().split())
        # Divide the range into smaller chunks (e.g., 100,000 numbers per chunk)
        chunk_size = 100000
        for chunk_start in range(start, end + 1, chunk_size):
            chunk_end = min(chunk_start + chunk_size - 1, end)
            # Yield prime numbers in this chunk
            for num in range(chunk_start, chunk_end + 1):
                if self.is_prime(num):
                    yield None, num

    def reducer(self, _, numbers):
        # Sum all prime numbers
        total = sum(numbers)
        yield "Sum of primes", total

    def is_prime(self, n):
        # Handle edge cases
        if n < 2:
            return False
        if n == 2:
            return True
        if n % 2 == 0:
            return False
        # Check odd divisors up to sqrt(n)
        for i in range(3, int(math.sqrt(n)) + 1, 2):
            if n % i == 0:
                return False
        return True

if __name__ == '__main__':
    MRPrimeSum.run()

Overwriting test.py


## Then we need to parse the input parameter. To run, we need to create a text file to contain input parameter, in this example, please input 1 10000000 and save the file

In [30]:
!python test.py test.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/test.root.20250519.170232.927514
Running step 1 of 1...
job output is in /tmp/test.root.20250519.170232.927514/output
Streaming final output from /tmp/test.root.20250519.170232.927514/output...
"Sum of primes"	37550402023
Removing temp directory /tmp/test.root.20250519.170232.927514...


# We will move on to run some exercises to further your knowledge :))

## First exercisesExercise 1: Reverse a Linked List (Easy)

Problem Description:
Given a sequence of linked lists represented as space-separated integers (e.g., "1 2 3 4" represents the list 1→2→3→4), write an MRJob program to reverse each linked list (e.g., output "4 3 2 1"). Each line of input represents one linked list. This introduces students to parallel processing of basic data structures.

Input Format:

    Each line contains space-separated integers representing a linked list (e.g., "1 2 3 4").
    Example input file:
    1 2 3 4
    5 6 7
    8 9
Output:

    Each reversed list as a space-separated string (e.g., "4 3 2 1").
    Example output:
    4 3 2 1
    7 6 5
    9 8

## Exercise 2: Detect Duplicate Elements in Arrays (Easy-Medium)

Problem Description:
Given a set of arrays represented as space-separated integers, write an MRJob program to detect if each array contains any duplicate elements. Output the array ID (line number) and whether it has duplicates (True/False). This introduces set operations and parallel processing of array-based data structures.

Input Format:

    Each line contains space-separated integers representing an array (e.g., "1 2 2 3").
    Example input file:
    1 2 2 3
    4 5 6
    7 7 7

Output:

    For each array, output the array ID (1-based line number) and True (has duplicates) or False (no duplicates).
    Example output:
    1 True
    2 False
    3 True

##Exercise 3: Compute Factorial for Large Numbers (Medium)

Problem Description:
Given a list of integers, compute the factorial of each number (n!). Since factorials grow large, use Python’s arbitrary-precision arithmetic. Output each number and its factorial. This exercise emphasizes parallel computation of a recursive algorithm and handling large data.

Input Format:

    Each line contains a single integer n (0 ≤ n ≤ 100).
    Example input file:
    5
    10
    0

Output:

    For each number, output the number and its factorial.
    Example output:
    0 1
    5 120
    10 3628800

## Exercise 4: Exercise 4: Find Shortest Paths in a Graph (Medium-Hard)

Problem Description:
Given a weighted directed graph represented as a list of edges (source, destination, weight), compute the shortest path distances from a fixed source node (e.g., node 1) to all other nodes using a MapReduce version of Dijkstra’s algorithm. This introduces parallel graph processing and iterative MapReduce jobs.

Input Format:

    Each line represents an edge: "source destination weight" (integers).
    Example input file (graph with nodes 1, 2, 3):
    1 2 4
    1 3 1
    2 3 2

Output:

    For each node (except the source), output the node and its shortest path distance from node 1.
    Example output (shortest paths from node 1):
    2 4
    3 1

#Exercise 5: Hard
Generate RSA Key Pair with Parallel Prime Search

Problem Description:
In RSA cryptography, used in SSL/TLS for secure communication, a key pair is generated by selecting two large prime numbers (p and q), computing the modulus (n = p * q), and deriving public and private exponents (e and d). Write an MRJob program to:

    Parallelize the search for two distinct 128-bit prime numbers using the Miller-Rabin primality test.
    Compute the RSA key pair (n, e, d) from the first valid prime pair found. Each mapper should generate random 128-bit numbers, test them for primality, and yield pairs of primes. The reducer selects the first valid pair. A second MapReduce job computes the RSA key components.

Input Format:

    A single line with the number of prime candidates to generate per mapper (e.g., "10").
    Example input file:
    10

Output:

    The RSA key components: modulus (n), public exponent (e), private exponent (d).
    Example output (values will vary due to randomness):
    n 12345678901234567890
    e 65537
    d 9876543210987654321

## Solution guide:
Data Structure: None explicitly (numbers and pairs), but the algorithm uses modular arithmetic and random number generation.
Algorithm:

    Miller-Rabin Primality Test: Probabilistic test to check if a number is prime (O(k * log^3 n) for k iterations).
    RSA Key Generation: Compute n = p * q, phi = (p-1)(q-1), and modular inverse for private key.

MapReduce:

    First Job:
        Mapper: Generates random 128-bit numbers, tests them with Miller-Rabin, and yields pairs of primes found.
        Reducer: Selects the first distinct prime pair (p, q).
    Second Job:
        Mapper: Computes RSA components (n, e, d) from the prime pair.
        Reducer: Outputs the RSA key components.

Parallelism:

    Each mapper independently generates and tests prime candidates, leveraging MRJob’s process-based parallelism.
    The reducer consolidates results, mimicking a real-world distributed key generation system.

The code template will be provided, but you must search algorithm and finished the rest to run the code

In [40]:
%%file exercise_5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import random
import os
import math

class MRRSAKeyGen(MRJob):
    def steps(self):
        return [
            MRStep(mapper_init=self.mapper_init_prime,
                   mapper=self.mapper_find_primes,
                   reducer=self.reducer_select_pair),
            MRStep(mapper=self.mapper_compute_rsa,
                   reducer=self.reducer_output_rsa)
        ]

    def mapper_init_prime(self):
        # Seed random number generator with process ID for uniqueness
        random.seed(os.getpid())
        self.bit_length = 128  # 128-bit primes for local machine
        self.k = 10  # Miller-Rabin iterations

    def mapper_find_primes(self, _, line):
        # Read number of candidates to generate
        num_candidates = int(line.strip())
        primes = []
        # Generate candidates and test for primality
        for _ in range(num_candidates):
            # Generate random 128-bit odd number
            candidate = random.getrandbits(self.bit_length) | 1
            if self.miller_rabin(candidate, self.k):
                primes.append(candidate)
        # Yield pairs of distinct primes
        for i in range(len(primes)):
            for j in range(i + 1, len(primes)):
                yield None, (primes[i], primes[j])

    def reducer_select_pair(self, _, pairs):
        # Select the first valid prime pair
            # Ensure distinct primes

    def mapper_compute_rsa(self, _, pair):
        # Compute RSA key components
        p, q = pair
        n = p * q
        phi = (p - 1) * (q - 1)
        e = 65537  # Standard public exponent
        # Compute private exponent d = e^(-1) mod phi
        d = self.mod_inverse(e, phi)
        if d:  # Ensure modular inverse exists
            yield None, (n, e, d)

    def reducer_output_rsa(self, _, rsa_components):
        # Output the first RSA key set

    def miller_rabin(self, n, k):

    def mod_inverse(self, a, m):
        # Compute modular inverse using extended GCD
        def extended_gcd(a, b):

if __name__ == '__main__':
    MRRSAKeyGen.run()

Overwriting exercise_5.py


In [44]:
!python exercise_5.py exercise_5.txt --output-dir="/content/output"

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 2...
Creating temp directory /tmp/exercise_5.root.20250519.172056.791342
Running step 2 of 2...
job output is in /content/output
Removing temp directory /tmp/exercise_5.root.20250519.172056.791342...
